In [ ]:
import os

os.environ.setdefault("HF_HOME", "/nfsd/lttm4/tesisti/shahrampour/hf")
os.environ.setdefault("HF_DATASETS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_datasets")
os.environ.setdefault("TRANSFORMERS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_transformers")

for k in ["HF_HOME","HF_DATASETS_CACHE","TRANSFORMERS_CACHE"]:
    os.makedirs(os.environ[k], exist_ok=True)

print("HF_HOME:", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE:", os.environ["HF_DATASETS_CACHE"])
print("TRANSFORMERS_CACHE:", os.environ["TRANSFORMERS_CACHE"])

## 1) Imports

In [ ]:
import numpy as np
import torch
import json
import pandas as pd
import matplotlib.pyplot as plt


from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoImageProcessor,
    AutoModelForImageClassification,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import set_seed

from peft import LoraConfig, get_peft_model, TaskType

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
RUN_NAME = "cifar100_5step_e15_10_5_15_V2"

RESULTS_DIR = os.path.join("results", RUN_NAME)
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
METRICS_DIR = os.path.join(RESULTS_DIR, "metrics")

STEP1_OUT = f"outputs/{RUN_NAME}/step1_scratch"
STEP1_FINAL_OUT = f"outputs/{RUN_NAME}/step1_scratch_final"
JOINT_OUT = f"outputs/{RUN_NAME}/joint_full"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(f"outputs/{RUN_NAME}", exist_ok=True)

## 2) Load CIFAR-100 (fine labels)

In [ ]:
from datasets import Image

dataset = load_dataset("cifar100")
dataset = dataset.rename_column("fine_label", "label")

dataset = dataset.cast_column("img", Image())

class_names = dataset["train"].features["label"].names
num_classes = len(class_names)

print("num_classes:", num_classes)
print("example keys:", dataset["train"][0].keys())
print("first 10 classes:", class_names[:10])

In [ ]:
def make_custom_eval_dataset(class_ids, remap_labels=True):
    test_ds = filter_dataset_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        test_ds = test_ds.map(remap)
    else:
        label2new = None
        new2orig = None

    test_ds.set_transform(preprocess_val)
    return test_ds, label2new, new2orig

## 3) Define incremental class splits (2/5/10 steps)

In [ ]:

num_steps = 5  

assert num_classes % num_steps == 0
classes_per_step = num_classes // num_steps

class_splits = [
    list(range(s * classes_per_step, (s + 1) * classes_per_step))
    for s in range(num_steps)
]

print("classes per step:", classes_per_step)
print("split sizes:", [len(x) for x in class_splits])
print("step0 sample:", class_splits[0][:10], "...", class_splits[0][-3:])

In [ ]:
first_step_classes = class_splits[0]
later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(class_splits[s])
all_classes = list(range(num_classes))

print("first step classes:", len(first_step_classes))
print("later step classes:", len(later_step_classes))
print("all classes:", len(all_classes))

## 4) Model + preprocessing

In [ ]:
# Requested model
model_checkpoint = "google/vit-base-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint, use_fast=True)

from torchvision import transforms
from PIL import Image
import numpy as np
import torch

size = image_processor.size
if isinstance(size, dict):
    H = size.get("height", 224)
    W = size.get("width", 224)
else:
    H = W = int(size)

train_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomCrop((H, W), padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

val_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")

    if isinstance(x, dict):
        if "array" in x:
            x = x["array"]
        elif "bytes" in x:
            import io
            return Image.open(io.BytesIO(x["bytes"])).convert("RGB")

    if isinstance(x, list):
        x = np.array(x, dtype=np.uint8)

    if isinstance(x, np.ndarray):
        arr = x.astype(np.uint8)
        arr = np.squeeze(arr)

        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)

        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))

        if not (arr.ndim == 3 and arr.shape[-1] in (1, 3)):
            raise TypeError(f"Unexpected image array shape after squeeze: {arr.shape}")

        if arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)

        return Image.fromarray(arr).convert("RGB")

    return x

def preprocess_train(ex):
    ex["pixel_values"] = [train_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def preprocess_val(ex):
    ex["pixel_values"] = [val_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def collate_fn(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([int(e["labels"]) for e in examples], dtype=torch.long)
    return {"pixel_values": pixel_values, "labels": labels}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": (preds == labels).mean().item()}

## 5) Build per-step datasets (step / cumulative / full)

In [1]:
def classes_for_step(step_idx: int) -> list[int]:
    return class_splits[step_idx]

def classes_for_cumulative(step_idx: int) -> list[int]:
    out = []
    for s in range(step_idx + 1):
        out.extend(class_splits[s])
    return out

def filter_by_classes(ds, class_ids):
    class_set = set(class_ids)
    ds = ds.with_format(None)
    return ds.filter(lambda x: int(x["label"]) in class_set)

def make_step_datasets(step_idx: int, split_type: str = "new_only", remap_labels: bool = False):
    """
    split_type:
      - 'new_only'   : only classes of this step
      - 'cumulative' : union of classes up to this step
      - 'full'       : all classes (100)
    """
    if split_type == "full":
        class_ids = list(range(num_classes))
    elif split_type == "cumulative":
        class_ids = classes_for_cumulative(step_idx)
    elif split_type == "new_only":
        class_ids = classes_for_step(step_idx)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")

    train_ds = filter_by_classes(dataset["train"], class_ids)
    test_ds  = filter_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        train_ds = train_ds.map(remap)
        test_ds  = test_ds.map(remap)
    else:
        label2new = {c: c for c in class_ids}
        new2orig = {c: c for c in class_ids}

    train_ds = train_ds.with_transform(preprocess_train)
    test_ds = test_ds.with_transform(preprocess_val)

    print(f"[Dataset] Step {step_idx+1} | split_type={split_type}")
    print(f"[Dataset] Classes: {class_ids[:5]} ... {class_ids[-5:]}")
    print(f"[Dataset] Train size: {len(train_ds)} | Test size: {len(test_ds)}")

    return train_ds, test_ds, label2new, new2orig, class_ids

eval_all = dataset["test"].with_transform(preprocess_val)

print("eval_all size:", len(eval_all))

## 6) Training recipes (reasonable settings)

In [ ]:
set_seed(42)

# SCRATCH_EPOCHS = 1
# LORA_EPOCHS    = 1
# FT_EPOCHS      = 1
# JOINT_EPOCHS   = 1

SCRATCH_EPOCHS = 15
LORA_EPOCHS    = 10
FT_EPOCHS      = 5
JOINT_EPOCHS   = 15

BATCH_SCRATCH = 8
ACCUM_SCRATCH = 2

BATCH_LORA    = 16
ACCUM_LORA    = 1

BATCH_FT      = 8
ACCUM_FT      = 2

LR_SCRATCH = 5e-5
LR_LORA    = 1e-4
LR_FT      = 3e-5
LR_JOINT   = 5e-5

WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
SCHED = "cosine"

USE_FP16 = torch.cuda.is_available()

results = []

In [ ]:
run_config = {
    "run_name": RUN_NAME,
    "model_checkpoint": model_checkpoint,
    "init_mode": "scratch",
    "num_classes": num_classes,
    "num_steps": num_steps,
    "classes_per_step": classes_per_step,
    "scratch_epochs": SCRATCH_EPOCHS,
    "lora_epochs": LORA_EPOCHS,
    "ft_epochs": FT_EPOCHS,
    "joint_epochs": JOINT_EPOCHS,
    "lr_scratch": LR_SCRATCH,
    "lr_lora": LR_LORA,
    "lr_ft": LR_FT,
    "lr_joint": LR_JOINT,
}
with open(os.path.join(METRICS_DIR, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

## 7) Step 1: train full ViT from scratch on step 0 classes

In [ ]:
step1_idx = 0
train_step1, test_step1, label2new_1, new2orig_1, class_ids_1 = make_step_datasets(
    step1_idx, split_type="new_only", remap_labels=False
)

print("Step1 original classes:", class_ids_1[:5], "...", class_ids_1[-5:])

config_step1 = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

model_step1 = AutoModelForImageClassification.from_config(config_step1)

args_step1 = TrainingArguments(
    output_dir=STEP1_OUT,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=SCRATCH_EPOCHS,
    learning_rate=LR_SCRATCH,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=SCHED,
    per_device_train_batch_size=BATCH_SCRATCH,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM_SCRATCH,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=True,
    max_grad_norm=1.0,
)

trainer_step1 = Trainer(
    model=model_step1,
    args=args_step1,
    train_dataset=train_step1,
    eval_dataset=test_step1,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

from transformers.utils.notebook import NotebookProgressCallback
trainer_step1.remove_callback(NotebookProgressCallback)

trainer_step1.train()

eval_step1 = trainer_step1.evaluate()

print({"eval_step1": eval_step1})

trainer_step1.save_model(STEP1_FINAL_OUT)
image_processor.save_pretrained(STEP1_FINAL_OUT)

In [ ]:
step1_log_df = pd.DataFrame(trainer_step1.state.log_history)
step1_log_df.to_csv(os.path.join(TABLES_DIR, "step1_log_history.csv"), index=False)

step1_metrics = {
    "experiment": "step1_scratch",
    "eval_loss": float(eval_step1.get("eval_loss", np.nan)),
    "eval_accuracy": float(eval_step1.get("eval_accuracy", np.nan)),
}

with open(os.path.join(METRICS_DIR, "step1_metrics.json"), "w") as f:
    json.dump(step1_metrics, f, indent=2)

results.append({
    "experiment": "step1_scratch",
    "method": "full_finetune",
    "step": 1,
    "eval_type": "current_step",
    "eval_accuracy": float(eval_step1.get("eval_accuracy", np.nan)),
    "eval_loss": float(eval_step1.get("eval_loss", np.nan)),
})

step1_log_df.tail()

In [ ]:
if "loss" in step1_log_df.columns:
    train_loss_df = step1_log_df[step1_log_df["loss"].notna()]
    if not train_loss_df.empty:
        plt.figure(figsize=(8,5))
        plt.plot(train_loss_df["step"], train_loss_df["loss"])
        plt.xlabel("Step")
        plt.ylabel("Train Loss")
        plt.title("Step 1 Train Loss")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "step1_train_loss.png"), dpi=200)
        plt.show()

if "eval_accuracy" in step1_log_df.columns:
    eval_df = step1_log_df[step1_log_df["eval_accuracy"].notna()]
    if not eval_df.empty:
        plt.figure(figsize=(8,5))
        plt.plot(eval_df["epoch"], eval_df["eval_accuracy"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Eval Accuracy")
        plt.title("Step 1 Eval Accuracy")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "step1_eval_accuracy.png"), dpi=200)
        plt.show()

## 8) Step 2: LoRA only (freeze backbone) on top of Step 1 model

In [ ]:
lora_results = []

base_model_dir = STEP1_FINAL_OUT

for step_idx in range(1, num_steps):
    train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
        step_idx, split_type="new_only", remap_labels=False
    )

    print(f"\n[LoRA] Step {step_idx+1}")
    print("Current step classes:", class_ids[:5], "...", class_ids[-5:])

    sample_train_labels = [int(train_step[i]["label"]) for i in range(min(200, len(train_step)))]
    sample_test_labels = [int(test_step[i]["label"]) for i in range(min(200, len(test_step)))]

    print("Train label range:", min(sample_train_labels), max(sample_train_labels))
    print("Test label range:", min(sample_test_labels), max(sample_test_labels))
    print("Num labels:", num_classes)

    assert min(sample_train_labels) >= 0 and max(sample_train_labels) < num_classes
    assert min(sample_test_labels) >= 0 and max(sample_test_labels) < num_classes

    base_model = AutoModelForImageClassification.from_pretrained(
        base_model_dir,
        num_labels=num_classes,
        id2label={i: str(i) for i in range(num_classes)},
        label2id={str(i): i for i in range(num_classes)},
        ignore_mismatched_sizes=True,
    )

    lora_config = LoraConfig(
        r=32,
        lora_alpha=64,
        lora_dropout=0.1,
        bias="none",
        target_modules=["query", "key", "value"],
        modules_to_save=["classifier"],
    )

    model_lora = get_peft_model(base_model, lora_config)
    model_lora.print_trainable_parameters()

    args_lora = TrainingArguments(
        output_dir=f"outputs/{RUN_NAME}/step_{step_idx+1}_lora",
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        num_train_epochs=LORA_EPOCHS,
        learning_rate=LR_LORA,
        weight_decay=0.01,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=BATCH_LORA,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=ACCUM_LORA,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none",
        max_grad_norm=1.0,
    )

    trainer_lora = Trainer(
        model=model_lora,
        args=args_lora,
        train_dataset=train_step,
        eval_dataset=test_step,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    trainer_lora.train()

    eval_current = trainer_lora.evaluate(test_step)

    pd.DataFrame(trainer_lora.state.log_history).to_csv(
        os.path.join(TABLES_DIR, f"step{step_idx+1}_lora_log_history.csv"),
        index=False
    )

    lora_results.append({
        "experiment": f"step{step_idx+1}_lora",
        "method": "lora",
        "step": step_idx + 1,
        "eval_type": "current_step",
        "eval_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
        "eval_loss": float(eval_current.get("eval_loss", np.nan)),
    })

    merged_model = trainer_lora.model.merge_and_unload()

    for attr in ["peft_config", "_hf_peft_config_loaded", "active_adapters"]:
        if hasattr(merged_model, attr):
            try:
                delattr(merged_model, attr)
            except Exception:
                pass

    step_lora_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_lora_final"
    os.makedirs(step_lora_dir, exist_ok=True)

    merged_model.save_pretrained(step_lora_dir, safe_serialization=False)
    image_processor.save_pretrained(step_lora_dir)

    base_model_dir = step_lora_dir

In [ ]:
def make_eval_dataset(class_ids):
    test_ds = filter_by_classes(dataset["test"], class_ids)
    test_ds = test_ds.with_transform(preprocess_val)
    return test_ds

In [ ]:
final_lora_dir = f"outputs/{RUN_NAME}/step_{num_steps}_lora_final"
final_lora_model = AutoModelForImageClassification.from_pretrained(final_lora_dir)

args_lora_eval = TrainingArguments(
    output_dir=f"outputs/{RUN_NAME}/final_lora_eval",
    remove_unused_columns=False,
    report_to="none",
    fp16=USE_FP16,
    per_device_eval_batch_size=32,
)

trainer_lora_final = Trainer(
    model=final_lora_model,
    args=args_lora_eval,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

final_seen_classes = classes_for_cumulative(num_steps - 1)

lora_test_first = make_eval_dataset(first_step_classes)
lora_test_later = make_eval_dataset(later_step_classes)
lora_test_seen = make_eval_dataset(final_seen_classes)

lora_final_first = trainer_lora_final.evaluate(lora_test_first)
lora_final_later = trainer_lora_final.evaluate(lora_test_later)
lora_final_seen = trainer_lora_final.evaluate(lora_test_seen)

print("LoRA final - first step:", lora_final_first)
print("LoRA final - later steps:", lora_final_later)
print("LoRA final - all seen:", lora_final_seen)

## 9) Baseline: full fine-tune instead of LoRA (Step 2)

In [ ]:
ft_results = []

base_model_dir = STEP1_FINAL_OUT

for step_idx in range(1, num_steps):
    train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
        step_idx, split_type="new_only", remap_labels=False
    )

    print(f"\n[FT] Step {step_idx+1}")
    print("Current step classes:", class_ids[:5], "...", class_ids[-5:])

    sample_train_labels = [int(train_step[i]["label"]) for i in range(min(200, len(train_step)))]
    sample_test_labels = [int(test_step[i]["label"]) for i in range(min(200, len(test_step)))]

    print("Train label range:", min(sample_train_labels), max(sample_train_labels))
    print("Test label range:", min(sample_test_labels), max(sample_test_labels))
    print("Num labels:", num_classes)

    assert min(sample_train_labels) >= 0 and max(sample_train_labels) < num_classes
    assert min(sample_test_labels) >= 0 and max(sample_test_labels) < num_classes

    ft_model = AutoModelForImageClassification.from_pretrained(
        base_model_dir,
        num_labels=num_classes,
        id2label={i: str(i) for i in range(num_classes)},
        label2id={str(i): i for i in range(num_classes)},
        ignore_mismatched_sizes=True,
    )

    for p in ft_model.parameters():
        p.requires_grad = True

    trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in ft_model.parameters())
    print("FT trainable params:", trainable, "/", total)

    args_ft = TrainingArguments(
        output_dir=f"outputs/{RUN_NAME}/step_{step_idx+1}_ft",
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        num_train_epochs=FT_EPOCHS,
        learning_rate=LR_FT,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=BATCH_FT,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=ACCUM_FT,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none",
        max_grad_norm=1.0,
    )

    trainer_ft = Trainer(
        model=ft_model,
        args=args_ft,
        train_dataset=train_step,
        eval_dataset=test_step,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    trainer_ft.train()

    eval_current = trainer_ft.evaluate(test_step)

    step_ft_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft_final"
    trainer_ft.save_model(step_ft_dir)
    image_processor.save_pretrained(step_ft_dir)
    base_model_dir = step_ft_dir

    pd.DataFrame(trainer_ft.state.log_history).to_csv(
        os.path.join(TABLES_DIR, f"step{step_idx+1}_ft_log_history.csv"),
        index=False
    )

    ft_results.append({
        "experiment": f"step{step_idx+1}_full_finetune",
        "method": "full_finetune",
        "step": step_idx + 1,
        "eval_type": "current_step",
        "eval_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
        "eval_loss": float(eval_current.get("eval_loss", np.nan)),
    })

In [ ]:
final_ft_dir = f"outputs/{RUN_NAME}/step_{num_steps}_ft_final"
final_ft_model = AutoModelForImageClassification.from_pretrained(final_ft_dir)

args_ft_eval = TrainingArguments(
    output_dir=f"outputs/{RUN_NAME}/final_ft_eval",
    remove_unused_columns=False,
    report_to="none",
    fp16=USE_FP16,
    per_device_eval_batch_size=32,
)

trainer_ft_final = Trainer(
    model=final_ft_model,
    args=args_ft_eval,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

ft_test_first = make_eval_dataset(first_step_classes)
ft_test_later = make_eval_dataset(later_step_classes)
ft_test_seen = make_eval_dataset(final_seen_classes)

ft_final_first = trainer_ft_final.evaluate(ft_test_first)
ft_final_later = trainer_ft_final.evaluate(ft_test_later)
ft_final_seen = trainer_ft_final.evaluate(ft_test_seen)

print("FT final - first step:", ft_final_first)
print("FT final - later steps:", ft_final_later)
print("FT final - all seen:", ft_final_seen)

## 10) Upper bound: joint training (full dataset)

In [ ]:
train_joint, test_joint, label2new_J, new2orig_J, class_ids_J = make_step_datasets(
    step_idx=0, split_type="full", remap_labels=False
)

config_joint = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

joint_model = AutoModelForImageClassification.from_config(config_joint)

args_joint = TrainingArguments(
    output_dir=JOINT_OUT,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=JOINT_EPOCHS,
    learning_rate=LR_JOINT,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=SCHED,
    per_device_train_batch_size=BATCH_FT,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM_FT,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    max_grad_norm=1.0,
)

trainer_joint = Trainer(
    model=joint_model,
    args=args_joint,
    train_dataset=train_joint,
    eval_dataset=test_joint,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

trainer_joint.train()

eval_joint = trainer_joint.evaluate()
print({"eval_joint_full": eval_joint})

In [ ]:
joint_log_df = pd.DataFrame(trainer_joint.state.log_history)
joint_log_df.to_csv(os.path.join(TABLES_DIR, "joint_log_history.csv"), index=False)

joint_metrics = {
    "experiment": "joint_full",
    "eval_loss": float(eval_joint.get("eval_loss", np.nan)),
    "eval_accuracy": float(eval_joint.get("eval_accuracy", np.nan)),
}

with open(os.path.join(METRICS_DIR, "joint_metrics.json"), "w") as f:
    json.dump(joint_metrics, f, indent=2)

joint_log_df.tail()

In [ ]:
joint_test_first = make_eval_dataset(first_step_classes)
joint_test_later = make_eval_dataset(later_step_classes)
joint_test_all = make_eval_dataset(all_classes)

joint_final_first = trainer_joint.evaluate(joint_test_first)
joint_final_later = trainer_joint.evaluate(joint_test_later)
joint_final_all = trainer_joint.evaluate(joint_test_all)

print("Joint final - first step:", joint_final_first)
print("Joint final - later steps:", joint_final_later)
print("Joint final - all classes:", joint_final_all)

## 11) Compare results (step test vs full test)

In [ ]:
def grab_acc(d):
    return float(d["eval_accuracy"]) if "eval_accuracy" in d else np.nan

def grab_loss(d):
    return float(d["eval_loss"]) if "eval_loss" in d else np.nan

results_rows = list(results)
results_rows.extend(lora_results)
results_rows.extend(ft_results)

results_rows.extend([
    {
        "experiment": "lora_final",
        "method": "lora",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(lora_final_first),
        "eval_loss": grab_loss(lora_final_first),
    },
    {
        "experiment": "lora_final",
        "method": "lora",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(lora_final_later),
        "eval_loss": grab_loss(lora_final_later),
    },
    {
        "experiment": "lora_final",
        "method": "lora",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(lora_final_seen),
        "eval_loss": grab_loss(lora_final_seen),
    },
    {
        "experiment": "ft_final",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(ft_final_first),
        "eval_loss": grab_loss(ft_final_first),
    },
    {
        "experiment": "ft_final",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(ft_final_later),
        "eval_loss": grab_loss(ft_final_later),
    },
    {
        "experiment": "ft_final",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(ft_final_seen),
        "eval_loss": grab_loss(ft_final_seen),
    },
    {
        "experiment": "joint_final",
        "method": "joint_upper_bound",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(joint_final_first),
        "eval_loss": grab_loss(joint_final_first),
    },
    {
        "experiment": "joint_final",
        "method": "joint_upper_bound",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(joint_final_later),
        "eval_loss": grab_loss(joint_final_later),
    },
    {
        "experiment": "joint_final",
        "method": "joint_upper_bound",
        "step": num_steps,
        "eval_type": "all_classes",
        "eval_accuracy": grab_acc(joint_final_all),
        "eval_loss": grab_loss(joint_final_all),
    },
])

results_df = pd.DataFrame(results_rows)
results_df.to_csv(os.path.join(TABLES_DIR, "results_summary.csv"), index=False)

results_df

In [ ]:
summary_lines = [
    "Experiment summary",
    "==================",
    "",
]

for _, row in results_df.iterrows():
    summary_lines.append(
        f"{row['experiment']} | method={row['method']} | step={row['step']} | "
        f"eval_type={row['eval_type']} | "
        f"eval_accuracy={row['eval_accuracy']:.4f} | eval_loss={row['eval_loss']:.4f}"
    )

with open(os.path.join(RESULTS_DIR, "summary.txt"), "w") as f:
    f.write("\n".join(summary_lines))

print("\n".join(summary_lines))